# 🛡️ TransferGuard: Detecting and Avoiding Negative Transfer in Lightweight Vision Models

> **Computer Vision Course Project**  
> **Group Members:**  
> - **Ghanishth Saini** (Roll: 24121)  
> - **Arvind Yadav** (Roll: 24059)  
> - **Aakash Jagtap** (Roll: 24157)  

---

### 📌 Overview
This notebook executes the full training and evaluation benchmark for **TransferGuard** on Google Colab GPU (T4/V100/A100).

**Hardware Acceleration Requirement:**
Go to **Runtime > Change runtime type > Hardware accelerator > T4 GPU** before executing the cells below.

## 1. Verify GPU Availability & Environment Setup

In [ ]:
!nvidia-smi
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")

## 2. Clone or Link Repository & Install Dependencies

In [ ]:
# Clone the repository and install dependencies
!git clone https://github.com/arvindyadav186/TransferGuard-Detecting-and-Avoiding-Negative-Transfer-in-Lightweight-Vision-Models.git
%cd TransferGuard-Detecting-and-Avoiding-Negative-Transfer-in-Lightweight-Vision-Models
!pip install -r requirements.txt
!pip install -e .


## 3. Layer-wise CKA Representation Alignment Analysis
We compute Centered Kernel Alignment (CKA) between ImageNet pre-trained representations and target domain representations across network stages.

In [ ]:
!python scripts/run_cka_analysis.py --dataset svhn --model mobilenet_v3_small --device auto --subset 500
from IPython.display import Image, display
display(Image("results/cka/cka_mobilenet_v3_small_svhn.png"))

## 4. Train Baselines & Proposed TransferGuard
We evaluate four key regimes on SVHN (Street View House Numbers):
1. **Train from Scratch:** Random initialization ({scratch}$)
2. **Standard Fine-Tuning:** End-to-end backpropagation across all layers ({transfer}$)
3. **Linear Probing:** Frozen backbone, only classification head updated
4. **TransferGuard (Ours):** CKA-guided adaptive layer freezing and routing

In [ ]:
# Baseline 1: Train from scratch
!python scripts/train_scratch.py --config configs/svhn_mobilenetv3.yaml --epochs 30 --device auto

In [ ]:
# Baseline 2: Standard Fine-Tuning (Observing Negative Transfer)
!python scripts/train_finetune.py --config configs/svhn_mobilenetv3.yaml --mode full --epochs 25 --device auto

In [ ]:
# Baseline 3: Linear Probing
!python scripts/train_finetune.py --config configs/svhn_mobilenetv3.yaml --mode linear_probe --epochs 20 --device auto

In [ ]:
# Proposed: TransferGuard Adaptive Routing
!python scripts/train_transferguard.py --config configs/svhn_mobilenetv3.yaml --threshold 0.65 --epochs 25 --device auto

## 5. Comprehensive Benchmark Evaluation & Transfer Gain Table
Computes Transfer Gain  = Acc_{transfer} - Acc_{scratch}$ and checks for negative transfer.

In [ ]:
!python scripts/evaluate_all.py

## 6. Generate 300 DPI Publication & Presentation Figures

In [ ]:
!python scripts/plot_results.py --output-dir results/figures

from IPython.display import Image, display
print("--- Figure 1: Transfer Gain Comparison (Positive vs Negative Transfer Zones) ---")
display(Image("results/figures/transfer_gain_comparison.png"))

print("--- Figure 2: Layer-wise CKA Representational Divergence Curves ---")
display(Image("results/figures/cka_divergence_curve.png"))

print("--- Figure 3: TransferGuard Routing Profile on SVHN ---")
display(Image("results/figures/transferguard_svhn_routing_profile.png"))

print("--- Figure 4: Training Convergence Trajectories ---")
display(Image("results/figures/training_convergence_comparison.png"))

## 7. Package Figures & Checkpoints for Presentation

In [ ]:
!zip -r transferguard_presentation_results.zip results/figures/ results/benchmark_summary.csv docs/
from google.colab import files
files.download("transferguard_presentation_results.zip")